# Step 8: Market Regime Analysis

This notebook implements objective market regime classification, computes strategy performance across distinct regimes, analyzes Markov transition matrices, and studies factor correlation to identify strategy edges.

In [ ]:
import os
import sys

# Insert project source root to system path for local imports
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.regime import MarketRegimeAnalyzer
from src.engine import backtest
from src.validation import split_dataset

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Modules imported successfully.")

## 1. Load Prices & Frozen Strategy Backtest Results

We reload Nifty 50 clean prices and run the optimized backtests for both strategies over the entire history (2013-2024).

In [ ]:
prices = pd.read_parquet("../data/processed/nifty50_clean.parquet")

# Load best optimized parameters discovered in validation phase
# Momentum: short_window=10, long_window=150
# Mean Reversion: window=30, entry_threshold=-2.0, exit_threshold=-0.5

from src.momentum import MomentumSignalGenerator
mom_gen = MomentumSignalGenerator(short_window=10, long_window=150)
mom_signals = mom_gen.generate_signals(prices)
mom_res = backtest(prices, mom_signals["Raw_Signal"])

from src.mean_reversion import MeanReversionSignalGenerator
mr_gen = MeanReversionSignalGenerator(window=30, entry_threshold=-2.0, exit_threshold=-0.5)
mr_signals = mr_gen.generate_signals(prices)
mr_res = backtest(prices, mr_signals["Raw_Signal"])

print("Backtests completed on whole history!")

## 2. Market Regime Detection

We instantiate the `MarketRegimeAnalyzer` to classify trend and volatility states and identify crash/recovery phases.

In [ ]:
analyzer = MarketRegimeAnalyzer(prices)
regimes_df = analyzer.detect_regimes()

print("Historical Regime Occurrences:")
print(regimes_df["Final_Regime"].value_counts())

print("\nStability statistics:")
for k, v in analyzer.stability_stats.items():
    if k != "By_Regime":
        print(f"  {k}: {v}")

## 3. Markov State Transition Probabilities

We print the matrix showing daily transition probabilities between market states.

In [ ]:
print("=== Transition Matrix ===")
display(analyzer.transition_matrix)

## 4. Performance breakdown by Regime

We calculate CAGR, Sharpe, and Drawdown for both strategies inside each distinct market state.

In [ ]:
mom_perf, mr_perf = analyzer.analyze_performance(mom_res, mr_res)

print("=== Momentum Performance by Regime ===")
display(mom_perf)

print("\n=== Mean Reversion Performance by Regime ===")
display(mr_perf)

## 5. Feature Importance Correlation & Transition Returns

We evaluate which market variables explain daily strategy returns, and check returns following regime transition changes.

In [ ]:
feat_imp = analyzer.run_feature_importance(mom_res)
print("=== Feature Importance Correlation ===")
print(feat_imp)

trans_returns = analyzer.run_transition_returns(mom_res)
print("\n=== Post-Transition Drift Returns ===")
display(trans_returns.head(10))

## 6. Plot Regime Analysis Visualizations (20 Charts)

In [ ]:
figures_dir = "../reports/figures"

analyzer.plot_regimes(
    mom_backtest_res=mom_res,
    mr_backtest_res=mr_res,
    mom_perf=mom_perf,
    mr_perf=mr_perf,
    feature_importance=feat_imp,
    output_dir=figures_dir
)

print("All 20 visualizations successfully generated and saved to reports/figures!")